# Week 63, Day 1: Descriptors

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w063d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class Loud:
    def __get__(self, obj, objtype=None):
        print("get called, obj is", "None" if obj is None else type(obj).__name__)
        return 42

class Box:
    size = Loud()

b = Box()
print(b.size)
print(Box.size)
b.__dict__["size"] = 7
print(b.size)

In [ ]:
class Celsius:
    "Reads and writes a temperature stored in Celsius, in Fahrenheit."
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj._celsius * 9 / 5 + 32

    def __set__(self, obj, value):
        obj._celsius = (value - 32) * 5 / 9

class Room:
    fahrenheit = Celsius()
    def __init__(self, celsius):
        self._celsius = celsius

r = Room(20)
print(r.fahrenheit)
r.fahrenheit = 50                  # calls Celsius.__set__(descriptor, r, 50)
print(r._celsius)
print(type(Room.fahrenheit).__name__)

In [ ]:
class Positive:
    def __set_name__(self, owner, name):
        self.name = name

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        try:
            return obj.__dict__[self.name]
        except KeyError:
            raise AttributeError(self.name) from None

    def __set__(self, obj, value):
        if value <= 0:
            raise ValueError(f"{self.name} must be positive, not {value}")
        obj.__dict__[self.name] = value

class Product:
    price = Positive()
    stock = Positive()
    def __init__(self, price, stock):
        self.price, self.stock = price, stock     # goes through Positive.__set__

tea, jam = Product(3, 10), Product(5, 2)
print(tea.price, jam.price, vars(tea))
try:
    tea.stock = 0
except ValueError as e:
    print(e)
print(Product.price.name)

In [ ]:
class Lazy:
    def __init__(self, func):
        self.func = func
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        value = self.func(obj)
        obj.__dict__[self.name] = value         # shadows this non-data descriptor from now on
        return value

class Report:
    def __init__(self, rows):
        self.rows = rows
    @Lazy
    def total(self):
        print("computing...")
        return sum(self.rows)

rep = Report([1, 2, 3])
print(rep.total)
print(rep.total)                                # no "computing..." this time
print("total" in vars(rep))

In [ ]:
class Greeter:
    def greet(self, name):
        return f"hello, {name}"

g = Greeter()
func = Greeter.__dict__["greet"]               # the plain function stored on the class
bound = func.__get__(g, Greeter)               # what g.greet does
print(bound("Ada"), bound.__self__ is g)
print(type(Greeter.__dict__["greet"]).__name__, hasattr(func, "__get__"), hasattr(func, "__set__"))

class Slotted:
    __slots__ = ("x",)
print(type(Slotted.__dict__["x"]).__name__)
print(hasattr(property, "__set__"), hasattr(classmethod, "__set__"))

> **Key idea.** A descriptor is an object stored on a class whose type defines __get__(self, obj, objtype), __set__(self, obj, value) or __delete__(self, obj); Python calls them for reads, writes and deletes of that attribute on instances. __get__ gets obj None when read through the class, and should return the descriptor itself. __set_name__(owner, name) tells the descriptor its name when the class is created. One descriptor serves every instance, so store values on the instance (obj.__dict__), never on the descriptor, and never with setattr under the same name. Data descriptors (with __set__ or __delete__) beat the instance __dict__; non-data descriptors (only __get__) lose to it, which is how caching works. Functions, property, staticmethod, classmethod and __slots__ are all descriptors.

### Exercise: A reusable check

Write a data descriptor `InRange(low, high)`: setting a value outside `low <= value <= high` raises `ValueError`, and setting a value that isn't an `int` or `float` (booleans don't count) raises `TypeError`. Use `__set_name__` and store values in the instance's `__dict__` under the attribute's own name. Reading an attribute that hasn't been set raises `AttributeError`; reading through the class returns the descriptor.

In [ ]:
class InRange:
    def __init__(self, low, high):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_range():\n    \"validating descriptors\"\n    class Patient:\n        temperature = InRange(30, 45)\n        age = InRange(0, 130)\n    p, q = Patient(), Patient()\n    p.temperature, q.temperature = 37, 39.5\n    assert p.temperature == 37 and q.temperature == 39.5 and vars(p) == {\"temperature\": 37}\n    for bad, err in ((50, ValueError), (29.9, ValueError), (\"37\", TypeError), (True, TypeError)):\n        try:\n            p.temperature = bad\n            assert False\n        except err:\n            pass\n    assert p.temperature == 37\n    try:\n        p.age\n        assert False\n    except AttributeError:\n        pass\n    assert isinstance(Patient.age, InRange) and Patient.age.low == 0")

### Exercise: Typed with a default

Write `Typed(kind, default=None)`: setting a value that isn't an instance of `kind` raises `TypeError` with the message `"<name> must be <kind name>"` (for example `"title must be str"`). Reading an attribute that was never set returns `default`. `del obj.attr` removes the stored value (so reads give the default again), and deleting one that isn't set raises `AttributeError`.

In [ ]:
class Typed:
    def __init__(self, kind, default=None):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_typed():\n    \"a typed field\"\n    class Book:\n        title = Typed(str, \"untitled\")\n        pages = Typed(int, 0)\n    b = Book()\n    assert b.title == \"untitled\" and b.pages == 0\n    b.title = \"Ak\u00e9\"\n    assert b.title == \"Ak\u00e9\" and Book().title == \"untitled\"\n    try:\n        b.pages = \"300\"\n        assert False\n    except TypeError as e:\n        assert str(e) == \"pages must be int\"\n    del b.title\n    assert b.title == \"untitled\"\n    try:\n        del b.title\n        assert False\n    except AttributeError:\n        pass\n    assert isinstance(Book.pages, Typed)")

### Exercise: Your own cached property

Write `cached(func)`, a **non-data** descriptor used as a decorator: the first read on an instance calls `func(obj)` and stores the result in the instance's `__dict__` under the attribute's name, so later reads don't call it. Assigning to the attribute replaces the cached value (that works for free, since it's non-data), `del` clears it so the next read recomputes, and the descriptor keeps the function's `__doc__`.

In [ ]:
class cached:
    def __init__(self, func):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_cached():\n    \"a non-data descriptor that caches\"\n    calls = []\n    class Data:\n        def __init__(self, xs):\n            self.xs = xs\n        @cached\n        def mean(self):\n            \"The average.\"\n            calls.append(1)\n            return sum(self.xs) / len(self.xs)\n    d = Data([1, 2, 3])\n    assert d.mean == 2 and d.mean == 2 and len(calls) == 1\n    e = Data([10])\n    assert e.mean == 10 and len(calls) == 2\n    d.xs.append(10)\n    del d.mean\n    assert d.mean == 4 and len(calls) == 3\n    d.mean = 99\n    assert d.mean == 99 and len(calls) == 3\n    assert Data.mean.__doc__ == \"The average.\" and not hasattr(cached, \"__set__\")")

### Exercise: Another name for an attribute

Write a data descriptor `Alias(target)` so that reading, setting and deleting the aliased attribute act on the attribute named `target` instead (use `getattr`, `setattr` and `delattr` on the target name). Then `Counted()`: a data descriptor storing values in the instance's `__dict__` that also counts, **per instance**, how many times the attribute has been set; `Counted.sets(obj, name)` is a static method returning that count for the named attribute (0 if never set).

In [ ]:
class Alias:
    def __init__(self, target):
        ___

class Counted:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_alias():\n    \"forwarding descriptors\"\n    class Point:\n        x = Counted()\n        y = Counted()\n        horizontal = Alias(\"x\")\n    p = Point()\n    p.x = 1\n    p.horizontal = 5\n    assert p.x == 5 and p.horizontal == 5\n    assert Counted.sets(p, \"x\") == 2 and Counted.sets(p, \"y\") == 0\n    q = Point()\n    q.x = 3\n    assert Counted.sets(q, \"x\") == 1 and p.x == 5\n    del p.horizontal\n    try:\n        p.x\n        assert False\n    except AttributeError:\n        pass\n    assert isinstance(Point.horizontal, Alias)")

### Exercise: Methods by hand

Functions become methods because they're descriptors. Write `bind(func, obj)`: return what `obj.<func>` would be if `func` were stored on `obj`'s class, using the function's `__get__` (don't use `types.MethodType`). Then write a descriptor `class_or_instance(func)`: reading it through an **instance** binds the instance as the first argument, and reading it through the **class** binds the class, so the same method works both ways.

In [ ]:
def bind(func, obj):
    ___

class class_or_instance:
    def __init__(self, func):
        ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_binding():\n    \"functions are descriptors\"\n    class Dog:\n        def __init__(self, name):\n            self.name = name\n    def speak(self, word):\n        return f\"{self.name} says {word}\"\n    rex = Dog(\"Rex\")\n    m = bind(speak, rex)\n    assert m(\"woof\") == \"Rex says woof\" and m.__self__ is rex and m.__func__ is speak\n    class Thing:\n        label = \"class\"\n        def __init__(self):\n            self.label = \"instance\"\n        @class_or_instance\n        def describe(who):\n            return who.label\n    assert Thing.describe() == \"class\" and Thing().describe() == \"instance\"\n    assert not uses(source, \"MethodType\")")

### Bug hunt: One value for everyone

`Score` stores the value on the descriptor itself, and there's only one descriptor for the whole class, so setting Ben's score changes Ada's too: this prints `20 20`. Store each value on its own instance so it prints `10 20`.

In [ ]:
class Score:
    def __get__(self, obj, objtype=None):
        return self.value
    def __set__(self, obj, value):
        self.value = value

class Player:
    score = Score()

ada, ben = Player(), Player()
ada.score = 10
ben.score = 20
print(ada.score, ben.score)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"per-instance storage\"\n    expect_output(\"10 20\")")

### Bug hunt: Asked through the class

Reading `Product.price` through the class calls `__get__` with `obj` set to `None`, and `None` has no `__dict__`, so this crashes with `AttributeError`. Return the descriptor itself when `obj` is `None` so it prints `Price`.

In [ ]:
class Price:
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, obj, objtype=None):
        return obj.__dict__[self.name]
    def __set__(self, obj, value):
        obj.__dict__[self.name] = value

class Product:
    price = Price()

print(type(Product.price).__name__)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"obj is None\"\n    expect_output(\"Price\")")

### Bug hunt: Setting itself forever

`__set__` stores the value with `setattr` under the attribute's own name, which calls `__set__` again, and again, until Python gives up with `RecursionError`. Store it directly in the instance's `__dict__` so this prints `42`.

In [ ]:
class Stored:
    def __set_name__(self, owner, name):
        self.name = name
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return obj.__dict__[self.name]
    def __set__(self, obj, value):
        setattr(obj, self.name, value)

class Answer:
    value = Stored()

a = Answer()
a.value = 42
print(a.value)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"no recursion\"\n    expect_output(\"42\")")

### Puzzle: A descriptor's reading half

Arrange the lines of a descriptor that learns its name and reads values from the instance. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # True or False: a descriptor only works when stored on the class
q = ___   # what obj is in __get__ when the attribute is read through the class
r = ___   # "data" or "non-data": a descriptor with only __get__
s = ___   # True or False: a non-data descriptor beats a value in the instance __dict__
t = ___   # when __set_name__ is called: "class creation" or "first use"

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p is True and q is None and r == \"non-data\", \"p, q and r\"\n    assert s is False and t == \"class creation\", \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a field framework

Build the core of a model library from descriptors. `Field(kind, *, default=REQUIRED, check=None)` is a data descriptor: setting checks `isinstance(value, kind)` (`TypeError`) and, if given, `check(value)` must be true (`ValueError`); values live in the instance's `__dict__`. `REQUIRED` is a sentinel.

`Model` is a base class:

- `fields()` (a class method) returns the names of all `Field`s, in definition order, including those inherited from base models (base fields first; a subclass may redefine a field, which keeps its original position).
- `__init__(**values)` sets every field from the keywords or its default; a missing required field raises `TypeError("missing field: name")`, an unknown keyword raises `TypeError("unknown field: name")`.
- `repr` is `Point(x=1, y=2)`, `==` compares class and field values, and `to_dict()` returns a dict of field values.

In [ ]:
REQUIRED = object()

class Field:
    def __init__(self, kind, *, default=REQUIRED, check=None):
        ___

class Model:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_fields():\n    \"descriptors as fields\"\n    class Point(Model):\n        x = Field(int)\n        y = Field(int, default=0)\n    p = Point(x=1, y=2)\n    assert p.x == 1 and p.y == 2 and Point(x=5).y == 0\n    assert Point.fields() == [\"x\", \"y\"] and repr(p) == \"Point(x=1, y=2)\"\n    assert p == Point(x=1, y=2) and p != Point(x=1) and p.to_dict() == {\"x\": 1, \"y\": 2}\n    for kwargs, message in (({}, \"missing field: x\"), ({\"x\": 1, \"z\": 3}, \"unknown field: z\")):\n        try:\n            Point(**kwargs)\n            assert False\n        except TypeError as e:\n            assert str(e) == message\n    try:\n        p.x = \"1\"\n        assert False\n    except TypeError:\n        pass\n    assert isinstance(Point.x, Field)\n\ndef test_inheritance_and_checks():\n    \"checks and inherited fields\"\n    class Item(Model):\n        name = Field(str)\n        price = Field(int, check=lambda v: v > 0)\n    class Discounted(Item):\n        percent = Field(int, default=10, check=lambda v: 0 <= v <= 100)\n        price = Field(int, default=1, check=lambda v: v > 0)\n    assert Discounted.fields() == [\"name\", \"price\", \"percent\"]\n    d = Discounted(name=\"tea\")\n    assert d.price == 1 and d.percent == 10 and repr(d) == \"Discounted(name='tea', price=1, percent=10)\"\n    try:\n        Item(name=\"jam\", price=0)\n        assert False\n    except ValueError:\n        pass\n    try:\n        d.percent = 101\n        assert False\n    except ValueError:\n        pass\n    assert Item(name=\"a\", price=1) != Discounted(name=\"a\", price=1)\n    assert d.to_dict() == {\"name\": \"tea\", \"price\": 1, \"percent\": 10}")

## Warm-up

Reading `b.size` didn't return the `Loud` object stored on the class: Python called its `__get__`, both through the instance and through the class. But once the instance had its own `size` in its `__dict__`, that value won and `__get__` wasn't called. The section on data and non-data descriptors explains why.

## The problem descriptors solve

Week 29's properties let a class check values as they're set:

```
class Product:
    @property
    def price(self):
        return self._price
    @price.setter
    def price(self, value):
        if value <= 0:
            raise ValueError("price must be positive")
        self._price = value
```

Now suppose `Product` also has `weight` and `stock`, and `Order` has `quantity`, all of which must be positive. Each needs the same eight lines, differing only in the name. Properties are written **per attribute**. A **descriptor** is a class whose instances **are** the reusable behaviour: write `Positive` once, then

```
class Product:
    price = Positive()
    weight = Positive()
```

## The descriptor protocol

A **descriptor** is any object whose class defines at least one of:

- `__get__(self, obj, objtype=None)`: called for `obj.attr`; `obj` is the instance, or `None` when the attribute is read from the class itself (`Product.price`), and `objtype` is the class;
- `__set__(self, obj, value)`: called for `obj.attr = value`;
- `__delete__(self, obj)`: called for `del obj.attr`.

It only works when the descriptor is stored as a **class attribute**. Python's attribute lookup (`object.__getattribute__`, tomorrow's subject) finds `price` in the class, sees that the object found has `__get__`, and calls it instead of returning it. A descriptor stored in an instance's `__dict__` is just an ordinary value.

By convention, `__get__` with `obj is None` returns the descriptor itself, so that `Product.price` gives you the descriptor to inspect.

## __set_name__ and where to store values

One descriptor object is shared by **every instance** of the class, so it must not keep a value on `self` (the descriptor): all instances would see the same value. Store each instance's value **on the instance**, usually in `obj.__dict__`, under a key the descriptor knows.

To know that key, define `__set_name__(self, owner, name)`. Python calls it once, when the class is created, for every descriptor in the class body, passing the class and the attribute name (`"price"`). Then:

- a **data** descriptor (one with `__set__`) can store under the **same name**, `obj.__dict__["price"]`, because, as you'll see next, a data descriptor wins over the instance `__dict__`, so the stored value never shadows it;
- or store under a private name such as `"_price"`.

Never write `setattr(obj, self.name, value)` inside `__set__` with the same name: that calls `__set__` again, forever.

## Data and non-data descriptors

The difference between the two kinds decides who wins when the **instance `__dict__`** has an entry with the same name:

- A **data descriptor** defines `__set__` or `__delete__`. It **beats** the instance `__dict__`: reading always goes through `__get__`. (Properties are data descriptors, even read-only ones: their `__set__` raises.)
- A **non-data descriptor** defines only `__get__`. The instance `__dict__` **beats** it: once the instance has its own value under that name, that value is returned and `__get__` isn't called.

Non-data descriptors make **caching** easy: compute in `__get__`, store the result in `obj.__dict__` under the same name, and every later read finds the stored value directly, with no call at all. That's exactly how `functools.cached_property` works. It's also why the warm-up's `Loud` (non-data) lost to `b.__dict__["size"]`.

## Descriptors you already use

- **Functions are non-data descriptors.** `def greet(self)` in a class stores a function; `function.__get__(obj, cls)` returns a **bound method** with `obj` filled in as `self`. That's the whole mechanism of methods, and why yesterday's `obj.method` made a new object each time.
- **`property`** is a data descriptor class; `@property` calls it with your getter, and `.setter` returns a new property with a setter added.
- **`staticmethod`** and **`classmethod`** are descriptors whose `__get__` returns the plain function or binds the class instead of the instance.
- **`__slots__`** creates one descriptor per slot on the class, storing values in fixed places inside the object.

## Practice

## Quiz

## Daily challenge

## Before you go

Open a Python shell and look at `type(vars(SomeClass)["some_method"])` for a class of yours, then call its `__get__` by hand. Tomorrow: exactly how `property` is built from the descriptor protocol, and the full attribute lookup order.